# Treino Colab: fine-tuning de gado aéreo (T2)
Fluxo principal: instalar, baixar dataset público, validar a base, treinar A e validar. A+B é uma opção separada que requer `gado.webm`.

In [ ]:
!git clone https://github.com/Edu4rdo-Gobatto/Cattle-counter.git 2>/dev/null || echo "repo ja existe"
%cd Cattle-counter
!mkdir -p models
!curl -L -o models/yolo11ft.pt https://raw.githubusercontent.com/xxilytoo/cattle_graze_model/main/yolo11ft.pt
!ls -lh models/yolo11ft.pt

In [ ]:
!pip install -q ultralytics supervision opencv-python tqdm datasets pillow
!python -c "import ultralytics; print(ultralytics.__version__)"

In [ ]:
!python download_dataset.py
!ls data/datasets/aerial-cows/ && cat data/datasets/aerial-cows/data.yaml

In [ ]:
!yolo val model=models/yolo11ft.pt data=data/datasets/aerial-cows/data.yaml split=test imgsz=640

In [ ]:
!python train.py --data data/datasets/aerial-cows/data.yaml --name a

In [ ]:
!yolo val model=runs/train/a/weights/best.pt data=data/datasets/aerial-cows/data.yaml split=test imgsz=640
!ls -lh runs/train/a/weights/best.pt runs/train/a/results.png

## Opção A+B (opcional)
Carregue o vídeo `gado.webm` para a sessão do Colab. O modelo gera os pseudo-rótulos automaticamente, sem caixas manuais.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!python pseudo_label.py --input gado.webm --output data/pseudo
!python train.py --name ab --pseudodata data/aerial-cows-ab.yaml

In [ ]:
!yolo val model=runs/train/ab/weights/best.pt data=data/datasets/aerial-cows/data.yaml split=test imgsz=640
!ls -lh runs/train/ab/weights/best.pt runs/train/ab/results.png